# Define variables

In [0]:
catalog = "workspace"
schema = "thinknyx"
volume = "thinknyx_vol"
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "rows.csv"
table_name = "nyx_table"
path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema

print(path_volume)
print(path_table)


# Loading csv file

In [0]:
dbutils.fs.cp(f"{download_url}", f"{path_volume}/{file_name}")

# Create a test dataframe

In [0]:
data = [[2021, "test", "Albany", "M", 42]]
df1 = spark.createDataFrame(data, schema="Year int, First_name string, country string, gender string, age int")
df1.show()

# Load data into dataframe from csv file

In [0]:
df_csv = spark.read.csv(f"{path_volume}/{file_name}", header=True, inferSchema=True, sep=",")
display(df_csv)

# View and interact with dataframe

In [0]:
df_csv.printSchema()

# Other Dataframe operations

In [0]:
df_csv.count()

In [0]:
df_csv.columns

In [0]:
df_csv.describe().show()

In [0]:
display(df_csv.summary())

# Filter dataframe

In [0]:
df_csv.filter("Year = 2011").show(5)

In [0]:
df_csv.filter(df_csv["Year"] == 2011).show(5)

In [0]:
from pyspark.sql.functions import desc

display(df_csv.select("Year", "First Name", "Count").orderBy(desc("Count")))

In [0]:
sub_df = df_csv.filter(df_csv["Year"] == 2011).select("County", "Count")
sub_df.show(5)

In [0]:
sub_df = sub_df.withColumnRenamed("Count", "Total")
sub_df.show()

In [0]:
sub_df.write.mode("overwrite").saveAsTable(f"{path_table}.{table_name}")

In [0]:
sub_df.write.format("json").mode("overwrite").save("/Volumes/workspace/thinknyx/thinknyx_vol/json_data")

In [0]:
display(spark.read.format("json").json("/Volumes/workspace/thinknyx/thinknyx_vol/json_data"))

In [0]:
display(spark.sql(f"select * from {path_table}.{table_name}"))